# Problem 1 — RNN, LSTM and GRU on sequences

Run this notebook from the `problem1/` folder (paths are relative). It uses the UCI HAR dataset in `data/har/UCI HAR Dataset/` and the generated adding problem. No pre-trained weights are used.

**Assumptions and deviations (stated once, here):**
- Hidden size `H = 64`, three seeds `0, 1, 2`, Adam with lr `1e-3`, batch `64`, gradient-norm clipping at `1.0` for every main run (so all three models get the same budget).
- Budget: HAR 30 epochs, adding problem 15 epochs. The machine has no GPU, so the epoch counts were chosen to fit a CPU run; they are the same for all three models.
- Model selection uses the **lowest validation loss epoch**. Test metrics are computed once, with that epoch's weights.
- Validation subjects are drawn per seed: `np.random.default_rng(seed).choice(train_subjects, 4, replace=False)`, as the brief specifies.
- The individual ablation (Task 5c) depends on the last digit of the student ID, which is not given to us. Both variants are run so either can be reported.
- `nn.RNN/LSTM/GRU` are used for the comparison experiments only. The cells in Task 2 are written from scratch and are only checked against the library.

In [ ]:
import random, time, copy, math
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

torch.set_num_threads(4)          # leave cores free for other jobs on this machine
DEVICE = torch.device("cpu")      # no CUDA here; see README

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

SEEDS = [0, 1, 2]
H = 64                            # hidden size of the main runs
HAR_EPOCHS = 30
ADD_EPOCHS = 15
BATCH = 64
LR = 1e-3
CLIP = 1.0                        # gradient-norm clipping in all main runs

FIG = Path("figures"); FIG.mkdir(exist_ok=True)
RESULTS_PATH = Path("results_problem1.csv")
COLS = ["problem","model","seed","seq_len","test_acc","macro_f1","test_mse","params",
        "sec_per_epoch","val_loss_nats","bits_per_char","notes"]
RESULTS = []

def save_results():
    pd.DataFrame(RESULTS, columns=COLS).to_csv(RESULTS_PATH, index=False)

def add_result(**kw):
    RESULTS.append({c: kw.get(c) for c in COLS}); save_results()

def n_params(m):
    return sum(p.numel() for p in m.parameters())

print("torch", torch.__version__, "| threads", torch.get_num_threads())

## Task 1 — Data pipelines

**HAR:** 9 channels × 128 steps, 6 classes. Official train/test partitions. The validation split is subject-wise (4 of the 21 training subjects per seed). Normalisation statistics come from the training part only.

In [ ]:
HAR_ROOT = Path("data/har/UCI HAR Dataset").resolve()
assert (HAR_ROOT / "train").is_dir(), "UCI HAR not found: see README for the download step"

SIGNALS = ["body_acc_x","body_acc_y","body_acc_z","body_gyro_x","body_gyro_y","body_gyro_z",
           "total_acc_x","total_acc_y","total_acc_z"]

def load_har(split):
    d = HAR_ROOT / split
    X = np.stack([np.loadtxt(d / "Inertial Signals" / f"{s}_{split}.txt") for s in SIGNALS], axis=-1)
    y = np.loadtxt(d / f"y_{split}.txt", dtype=np.int64) - 1        # labels 0..5
    subj = np.loadtxt(d / f"subject_{split}.txt", dtype=np.int64)
    return X.astype(np.float32), y, subj                           # X: (N, 128, 9)

X_tr_all, y_tr_all, subj_tr = load_har("train")
X_te, y_te, subj_te = load_har("test")
print("train", X_tr_all.shape, "test", X_te.shape)

def make_har_data(seed):
    subjects = np.unique(subj_tr)
    val_subjects = np.random.default_rng(seed).choice(subjects, 4, replace=False)
    va = np.isin(subj_tr, val_subjects)
    tr = ~va
    mu = X_tr_all[tr].mean(axis=(0, 1), keepdims=True)          # per channel, training part only
    sd = X_tr_all[tr].std(axis=(0, 1), keepdims=True) + 1e-6
    norm = lambda X: ((X - mu) / sd).astype(np.float32)
    assert set(subj_tr[tr]).isdisjoint(val_subjects), "subject leakage: train/val"
    assert set(val_subjects).isdisjoint(subj_te), "subject leakage: val/test"
    return dict(Xtr=norm(X_tr_all[tr]), ytr=y_tr_all[tr], Xva=norm(X_tr_all[va]), yva=y_tr_all[va],
                Xte=norm(X_te), yte=y_te, val_subjects=val_subjects.tolist())

HAR_DATA = {s: make_har_data(s) for s in SEEDS}
for s in SEEDS:
    D = HAR_DATA[s]
    print(f"seed {s}: val subjects {D['val_subjects']} | train {D['Xtr'].shape} | val {D['Xva'].shape} | test {D['Xte'].shape}")

In [ ]:
NAMES = ["Walking","Upstairs","Downstairs","Sitting","Standing","Laying"]
tr_counts = np.bincount(y_tr_all, minlength=6)
te_counts = np.bincount(y_te, minlength=6)
x = np.arange(6)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(x - 0.2, tr_counts, 0.4, label="train (21 subjects)")
ax.bar(x + 0.2, te_counts, 0.4, label="test (9 subjects)")
ax.set_xticks(x, NAMES); ax.set_ylabel("windows"); ax.set_title("HAR class balance"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "class_balance.png", dpi=160); plt.show()
print(dict(zip(NAMES, tr_counts.tolist())))

**Adding problem:** the generator is the one given in the brief, so the data are identical for every student. Its test is an assert cell placed directly after it.

In [ ]:
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

def test_adding_generator():
    for T in [50, 100, 200, 400]:
        x, y = make_adding(2000, T, seed=0)
        m = x[:, :, 1]
        assert np.all(m.sum(1) == 2), "each sequence needs exactly two markers"
        assert np.all(m[:, :T // 2].sum(1) == 1), "one marker in the first half"
        assert np.all(m[:, T // 2:].sum(1) == 1), "one marker in the second half"
        assert np.allclose(y, (x[:, :, 0] * m).sum(1), atol=1e-6), "target must equal the sum of marked values"
    print("adding-problem generator tests passed")

test_adding_generator()

## Task 2 — RNN, LSTM and GRU cells from scratch

Each cell uses only matrix operations. Gate order follows PyTorch: LSTM `(i, f, g, o)`, GRU `(r, z, n)`. The GRU candidate uses `r * (W_hn h + b_hn)`, which is PyTorch's convention. Copying the library weights into these cells must reproduce the library outputs within 1e-5.

In [ ]:
def _affine(x, W, b):
    # x: (B, in), W: (out, in), b: (out,)  ->  (B, out)
    return x @ W.T + b

def _init(*shape):
    k = 1 / math.sqrt(shape[0] if len(shape) == 1 else H)
    return nn.Parameter(torch.empty(*shape).uniform_(-k, k))

class MyRNNCell(nn.Module):
    """h_t = tanh(W_ih x_t + b_ih + W_hh h_{t-1} + b_hh).   x: (B, d), h: (B, h)"""
    def __init__(self, d, h):
        super().__init__()
        self.weight_ih = nn.Parameter(torch.empty(h, d).uniform_(-1/math.sqrt(h), 1/math.sqrt(h)))  # (h, d)
        self.weight_hh = nn.Parameter(torch.empty(h, h).uniform_(-1/math.sqrt(h), 1/math.sqrt(h)))  # (h, h)
        self.bias_ih = nn.Parameter(torch.empty(h).uniform_(-1/math.sqrt(h), 1/math.sqrt(h)))
        self.bias_hh = nn.Parameter(torch.empty(h).uniform_(-1/math.sqrt(h), 1/math.sqrt(h)))
    def forward(self, x, h_prev):
        return torch.tanh(_affine(x, self.weight_ih, self.bias_ih) + _affine(h_prev, self.weight_hh, self.bias_hh))

class MyLSTMCell(nn.Module):
    """Gates (i, f, g, o). W_ih: (4h, d), W_hh: (4h, h)."""
    def __init__(self, d, h):
        super().__init__(); k = 1 / math.sqrt(h)
        self.weight_ih = nn.Parameter(torch.empty(4*h, d).uniform_(-k, k))   # (4h, d)
        self.weight_hh = nn.Parameter(torch.empty(4*h, h).uniform_(-k, k))   # (4h, h)
        self.bias_ih = nn.Parameter(torch.empty(4*h).uniform_(-k, k))
        self.bias_hh = nn.Parameter(torch.empty(4*h).uniform_(-k, k))
    def forward(self, x, state):
        h_prev, c_prev = state                                              # each (B, h)
        gates = _affine(x, self.weight_ih, self.bias_ih) + _affine(h_prev, self.weight_hh, self.bias_hh)  # (B, 4h)
        i, f, g, o = gates.chunk(4, dim=-1)                                 # each (B, h)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g                                              # (B, h)
        h = o * torch.tanh(c)                                               # (B, h)
        return h, c

class MyGRUCell(nn.Module):
    """Gates (r, z, n). W_ih: (3h, d), W_hh: (3h, h)."""
    def __init__(self, d, h):
        super().__init__(); k = 1 / math.sqrt(h)
        self.weight_ih = nn.Parameter(torch.empty(3*h, d).uniform_(-k, k))   # (3h, d)
        self.weight_hh = nn.Parameter(torch.empty(3*h, h).uniform_(-k, k))   # (3h, h)
        self.bias_ih = nn.Parameter(torch.empty(3*h).uniform_(-k, k))
        self.bias_hh = nn.Parameter(torch.empty(3*h).uniform_(-k, k))
    def forward(self, x, h_prev):
        gi = _affine(x, self.weight_ih, self.bias_ih)                       # (B, 3h)
        gh = _affine(h_prev, self.weight_hh, self.bias_hh)                  # (B, 3h)
        i_r, i_z, i_n = gi.chunk(3, dim=-1)
        h_r, h_z, h_n = gh.chunk(3, dim=-1)
        r = torch.sigmoid(i_r + h_r)
        z = torch.sigmoid(i_z + h_z)
        n = torch.tanh(i_n + r * h_n)
        return (1 - z) * n + z * h_prev                                     # (B, h)

class Unrolled(nn.Module):
    """Runs a cell over x: (B, T, d). Keeps every h_t in self.states so that
    retain_grad() on them gives the total gradient dL/dh_t (used in Task 5a)."""
    def __init__(self, cell, h, lstm=False):
        super().__init__(); self.cell = cell; self.h = h; self.lstm = lstm; self.states = []
    def forward(self, x):
        B, T, _ = x.shape
        h0 = x.new_zeros(B, self.h)
        state = (h0, h0.clone()) if self.lstm else h0
        self.states, outs = [], []
        for t in range(T):
            state = self.cell(x[:, t], state)
            h_t = state[0] if self.lstm else state
            outs.append(h_t); self.states.append(h_t)
        return torch.stack(outs, 1), state                                  # (B, T, h), final state

def my_model(kind, d, h):
    if kind == "RNN":  return Unrolled(MyRNNCell(d, h), h)
    if kind == "LSTM": return Unrolled(MyLSTMCell(d, h), h, lstm=True)
    return Unrolled(MyGRUCell(d, h), h)

def lib_layer(kind, d, h):
    return {"RNN": lambda: nn.RNN(d, h, batch_first=True),
            "LSTM": lambda: nn.LSTM(d, h, batch_first=True),
            "GRU": lambda: nn.GRU(d, h, batch_first=True)}[kind]()

def copy_from_library(mine, lib):
    # the library stores the same tensors under *_l0 names, in the same gate order
    c = mine.cell
    c.weight_ih.data.copy_(lib.weight_ih_l0.data); c.weight_hh.data.copy_(lib.weight_hh_l0.data)
    c.bias_ih.data.copy_(lib.bias_ih_l0.data);     c.bias_hh.data.copy_(lib.bias_hh_l0.data)
    return mine

In [ ]:
def verify_cell(kind, d=9, h=64, B=32, T=128, tol=1e-5):
    set_seed(0)
    lib = lib_layer(kind, d, h).eval()
    mine = copy_from_library(my_model(kind, d, h), lib).eval()
    x = torch.randn(B, T, d)                                    # random batch
    with torch.no_grad():
        out_lib, st_lib = lib(x)
        out_my, st_my = mine(x)
    d_out = (out_lib - out_my).abs().max().item()
    if kind == "LSTM":
        d_state = max((st_lib[0][0] - st_my[0]).abs().max().item(),
                      (st_lib[1][0] - st_my[1]).abs().max().item())
    else:
        d_state = (st_lib[0] - st_my).abs().max().item()
    print(f"{kind:4s}: max |output diff| = {d_out:.2e}   max |final state diff| = {d_state:.2e}")
    assert d_out < tol and d_state < tol, f"{kind} does not match the library"

for k in ["RNN", "LSTM", "GRU"]:
    verify_cell(k)
print("All three cells match nn.RNN / nn.LSTM / nn.GRU within 1e-5.")

**Parameter counts for d = 9, h = 64, by hand.** With one bias vector, the RNN has `h(d+h) + h = 64·73 + 64 = 4,736`. The LSTM has 4 such blocks (`4·4,736 = 18,944`) and the GRU has 3 (`3·4,736 = 14,208`). PyTorch keeps two bias vectors (`bias_ih` and `bias_hh`), which adds `h`, `4h` and `3h` respectively: `4,800`, `19,200` and `14,400`. The cells above also hold two bias vectors, so they match PyTorch's count.

In [ ]:
d, h = 9, 64
hand_one_bias = {"RNN": h*(d+h)+h, "LSTM": 4*(h*(d+h)+h), "GRU": 3*(h*(d+h)+h)}
lib_counts = {k: n_params(lib_layer(k, d, h)) for k in ["RNN","LSTM","GRU"]}
my_counts = {k: n_params(my_model(k, d, h)) for k in ["RNN","LSTM","GRU"]}
print(pd.DataFrame({"hand (one bias)": hand_one_bias, "my cells": my_counts, "nn.* layer": lib_counts}))
assert hand_one_bias == {"RNN": 4736, "LSTM": 18944, "GRU": 14208}
assert lib_counts == {"RNN": 4800, "LSTM": 19200, "GRU": 14400}
assert my_counts == lib_counts
print("Parameter counts reconcile: hand derivation + one extra bias vector per gate block = PyTorch.")

## Task 3 — HAR classification

Each model reads the 128 steps and classifies from the last hidden state with a linear head. `params` in `results_problem1.csv` is the whole model, including the head. The recurrent-layer count is printed separately.

In [ ]:
class RecClassifier(nn.Module):
    """Recurrent layer (library) -> last hidden state -> linear head."""
    def __init__(self, kind, d_in, h, n_out):
        super().__init__()
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind](d_in, h, batch_first=True)
        self.head = nn.Linear(h, n_out)
    def forward(self, x):                # x: (B, T, d_in)
        out, _ = self.rnn(x)             # out: (B, T, h)
        return self.head(out[:, -1])     # (B, n_out)

def set_lstm_forget_bias(model, value):
    # PyTorch LSTM gate order is (i, f, g, o): the forget gate is the second quarter
    hh = model.rnn.hidden_size
    with torch.no_grad():
        model.rnn.bias_ih_l0[hh:2*hh].fill_(value)
        model.rnn.bias_hh_l0[hh:2*hh].fill_(0.0)     # effective forget bias = value

CE_LOSS = lambda p, y: F.cross_entropy(p, y)
MSE_LOSS = lambda p, y: F.mse_loss(p.squeeze(-1), y)

@torch.no_grad()
def predict(model, data, loss_fn):
    model.eval()
    X, y = torch.tensor(data[0]), torch.tensor(data[1])
    outs = torch.cat([model(X[i:i+512]) for i in range(0, len(X), 512)])
    loss = loss_fn(outs, y).item()
    preds = outs.argmax(-1).numpy() if loss_fn is CE_LOSS else outs.squeeze(-1).numpy()
    return loss, preds

def run_model(model, tr, va, loss_fn, epochs, clip=CLIP, seed=0):
    """Adam training; keep the epoch with the lowest validation loss (test data is never used)."""
    set_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    Xtr, ytr = torch.tensor(tr[0]), torch.tensor(tr[1])
    hist = {"train": [], "val": [], "epoch_sec": []}
    best = {"val": float("inf"), "state": None, "epoch": -1}
    for ep in range(epochs):
        model.train(); t0 = time.time(); total = 0.0
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), BATCH):
            idx = perm[i:i+BATCH]
            loss = loss_fn(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward()
            if clip: nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step(); total += loss.item() * len(idx)
        hist["epoch_sec"].append(time.time() - t0)
        hist["train"].append(total / len(Xtr))
        vl, _ = predict(model, va, loss_fn)
        hist["val"].append(vl)
        if vl < best["val"]:
            best = {"val": vl, "state": copy.deepcopy(model.state_dict()), "epoch": ep}
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return model, hist, best

HAR_RES = {}
def train_har(kind, hh, s, clip=CLIP, note=""):
    D = HAR_DATA[s]
    set_seed(s)
    model = RecClassifier(kind, 9, hh, 6)
    n_par = n_params(model)
    model, hist, best = run_model(model, (D["Xtr"], D["ytr"]), (D["Xva"], D["yva"]), CE_LOSS, HAR_EPOCHS, clip=clip, seed=s)
    _, pred = predict(model, (D["Xte"], D["yte"]), CE_LOSS)
    _, pred_va = predict(model, (D["Xva"], D["yva"]), CE_LOSS)
    r = dict(model=model, hist=hist, best_epoch=best["epoch"], params=n_par, pred=pred,
             acc=accuracy_score(D["yte"], pred), f1=f1_score(D["yte"], pred, average="macro"),
             val_f1=f1_score(D["yva"], pred_va, average="macro"), sec=float(np.mean(hist["epoch_sec"])))
    add_result(problem=1, model=kind, seed=s, seq_len=128, test_acc=r["acc"], macro_f1=r["f1"],
               params=n_par, sec_per_epoch=r["sec"],
               notes=f"HAR; H={hh}; epochs={HAR_EPOCHS}; clip={clip}; best_epoch={best['epoch']}; val_macro_f1={r['val_f1']:.4f}{note}")
    return r

for kind in ["RNN", "LSTM", "GRU"]:
    for s in SEEDS:
        HAR_RES[(kind, H, s)] = train_har(kind, H, s)
        r = HAR_RES[(kind, H, s)]
        print(f"HAR {kind:4s} seed {s}: test acc {r['acc']:.4f}  macro-F1 {r['f1']:.4f}  best epoch {r['best_epoch']}  {r['sec']:.2f}s/epoch")

In [ ]:
rows = []
for kind in ["RNN", "LSTM", "GRU"]:
    rs = [HAR_RES[(kind, H, s)] for s in SEEDS]
    acc = np.array([r["acc"] for r in rs]); f1s = np.array([r["f1"] for r in rs])
    rows.append({"model": kind, "params (total)": rs[0]["params"],
                 "test acc (%)": f"{100*acc.mean():.2f} ± {100*acc.std(ddof=1):.2f}",
                 "test macro-F1": f"{f1s.mean():.4f} ± {f1s.std(ddof=1):.4f}",
                 "sec/epoch": round(float(np.mean([r['sec'] for r in rs])), 2),
                 "val macro-F1 (mean)": round(float(np.mean([r["val_f1"] for r in rs])), 4)})
har_table = pd.DataFrame(rows)
display(har_table)
har_table.to_csv(FIG.parent / "har_summary_table.csv", index=False)

best_kind = max(["RNN","LSTM","GRU"], key=lambda k: np.mean([HAR_RES[(k, H, s)]["val_f1"] for s in SEEDS]))
print("Best HAR model (by validation macro-F1):", best_kind)
cm = sum(confusion_matrix(HAR_DATA[s]["yte"], HAR_RES[(best_kind, H, s)]["pred"], labels=range(6)) for s in SEEDS)
disp = ConfusionMatrixDisplay(cm, display_labels=NAMES)
disp.plot(cmap="Blues", xticks_rotation=45, values_format="d")
plt.title(f"{best_kind}: test confusion matrix (3 seeds summed)")
plt.tight_layout(); plt.savefig(FIG / "har_confusion_best.png", dpi=160); plt.show()

In [ ]:
# Classical baselines from the earlier notebook, for reference (not the recurrent comparison)
classical = Path("test_results.csv")
if classical.exists():
    print("Classical best (SVM) test metrics from classical_baselines.ipynb:")
    display(pd.read_csv(classical).round(4))

## Task 4 — Adding problem, T = 50, 100, 200, 400

Three models × three seeds × four lengths. The constant-prediction baseline (predict 1.0) is printed for each T. A run is *solved* if its test MSE is below 0.05.

In [ ]:
TS = [50, 100, 200, 400]
ADD_DATA = {T: (make_adding(10000, T, seed=0), make_adding(1000, T, seed=1), make_adding(1000, T, seed=2)) for T in TS}
BASELINE = {}
for T in TS:
    tr, va, te = ADD_DATA[T]
    BASELINE[T] = float(np.mean((te[1] - 1.0) ** 2))
    print(f"T={T:3d}: constant-1.0 baseline test MSE = {BASELINE[T]:.4f}")

ADD_RES = {}
INIT_SNAPSHOT = {}
def train_adding(kind, T, s, hh=H, clip=CLIP, forget_bias=None, note=""):
    tr, va, te = ADD_DATA[T]
    set_seed(s)
    model = RecClassifier(kind, 2, hh, 1)
    if forget_bias is not None and kind == "LSTM":
        set_lstm_forget_bias(model, forget_bias)
    if T == 100 and s == 0 and forget_bias is None and clip == CLIP and hh == H:
        INIT_SNAPSHOT[kind] = copy.deepcopy(model)          # untrained weights for Task 5a
    n_par = n_params(model)
    model, hist, best = run_model(model, tr, va, MSE_LOSS, ADD_EPOCHS, clip=clip, seed=s)
    test_mse, _ = predict(model, te, MSE_LOSS)
    solved = bool(test_mse < 0.05)
    r = dict(model=model, hist=hist, mse=test_mse, solved=solved, params=n_par, best_epoch=best["epoch"])
    add_result(problem=1, model=kind, seed=s, seq_len=T, test_mse=test_mse, params=n_par,
               sec_per_epoch=float(np.mean(hist["epoch_sec"])),
               notes=f"adding; H={hh}; epochs={ADD_EPOCHS}; clip={clip}; best_epoch={best['epoch']}; solved={solved}; baseline_mse={BASELINE[T]:.4f}{note}")
    return r

for T in TS:
    for kind in ["RNN", "LSTM", "GRU"]:
        for s in SEEDS:
            ADD_RES[(kind, T, s)] = train_adding(kind, T, s)
            r = ADD_RES[(kind, T, s)]
            print(f"T={T:3d} {kind:4s} seed {s}: test MSE {r['mse']:.4f}  {'solved' if r['solved'] else 'not solved'}  ({np.mean(r['hist']['epoch_sec']):.1f}s/epoch)")

In [ ]:
rows = []
for kind in ["RNN","LSTM","GRU"]:
    for T in TS:
        m = np.array([ADD_RES[(kind, T, s)]["mse"] for s in SEEDS])
        rows.append({"model": kind, "T": T,
                     "test MSE": f"{m.mean():.4f} ± {m.std(ddof=1):.4f}",
                     "solved (of 3)": sum(ADD_RES[(kind, T, s)]["solved"] for s in SEEDS),
                     "baseline MSE": round(BASELINE[T], 4)})
add_table = pd.DataFrame(rows)
display(add_table)
add_table.to_csv(FIG.parent / "adding_summary_table.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for kind in ["RNN","LSTM","GRU"]:
    mu = [np.mean([ADD_RES[(kind, T, s)]["mse"] for s in SEEDS]) for T in TS]
    sd = [np.std([ADD_RES[(kind, T, s)]["mse"] for s in SEEDS], ddof=1) for T in TS]
    ax.errorbar(TS, mu, yerr=sd, marker="o", capsize=3, label=kind)
ax.plot(TS, [BASELINE[T] for T in TS], "k--", label="predict 1.0 baseline")
ax.axhline(0.05, color="grey", ls=":", label="solved threshold (0.05)")
ax.set_xscale("log"); ax.set_xticks(TS, [str(T) for T in TS])
ax.set_xlabel("sequence length T"); ax.set_ylabel("test MSE (mean ± std, 3 seeds)")
ax.set_title("Adding problem: test MSE against T"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "adding_mse_vs_T.png", dpi=160); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ep = np.arange(1, ADD_EPOCHS + 1)
for kind in ["RNN","LSTM","GRU"]:
    tr_c = np.array([ADD_RES[(kind, 200, s)]["hist"]["train"] for s in SEEDS])
    va_c = np.array([ADD_RES[(kind, 200, s)]["hist"]["val"] for s in SEEDS])
    for ax, c in zip(axes, [tr_c, va_c]):
        ax.plot(ep, c.mean(0), label=kind)
        ax.fill_between(ep, c.mean(0) - c.std(0, ddof=1), c.mean(0) + c.std(0, ddof=1), alpha=0.2)
axes[0].set_title("training MSE, T = 200"); axes[1].set_title("validation MSE, T = 200")
for ax in axes:
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "adding_learning_curves_T200.png", dpi=160); plt.show()

## Task 5 — Gradients, ablations, hidden-size sweep

**(a)** Gradient norms ‖∂L/∂h_t‖ for T = 100, before and after training (seed 0 weights from Task 4). The profile uses the from-scratch unrolled cells with the trained library weights copied in, so that every h_t receives its total gradient. The profile is averaged over 256 held-out sequences.

In [ ]:
Xg, yg = make_adding(256, 100, seed=3)
Xg_t, yg_t = torch.tensor(Xg), torch.tensor(yg)

def grad_profile(my, head, X, y):
    my.zero_grad()
    out, _ = my(X)                                       # out: (B, T, h)
    for h_t in my.states: h_t.retain_grad()
    loss = F.mse_loss(head(out[:, -1]).squeeze(-1), y)
    loss.backward()
    norms = torch.stack([h_t.grad.norm(dim=-1) for h_t in my.states], 1)   # (B, T)
    return norms.mean(0).detach().numpy()

PROF = {}
for kind in ["RNN","LSTM","GRU"]:
    before = INIT_SNAPSHOT[kind]
    after = ADD_RES[(kind, 100, 0)]["model"]
    PROF[(kind, "before")] = grad_profile(copy_from_library(my_model(kind, 2, H), before.rnn), before.head, Xg_t, yg_t)
    PROF[(kind, "after")] = grad_profile(copy_from_library(my_model(kind, 2, H), after.rnn), after.head, Xg_t, yg_t)
    # the from-scratch cell must reproduce the trained library layer on these weights
    mine = copy_from_library(my_model(kind, 2, H), after.rnn).eval()
    with torch.no_grad():
        diff = (mine(Xg_t)[0] - after.rnn(Xg_t)[0]).abs().max().item()
    assert diff < 1e-5, f"{kind}: trained-weight mismatch {diff}"

t = np.arange(1, 101)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, when in zip(axes, ["before", "after"]):
    for kind in ["RNN","LSTM","GRU"]:
        ax.semilogy(t, PROF[(kind, when)], label=kind)
    ax.set_title(f"‖∂L/∂h_t‖ at T = 100, {when} training"); ax.set_xlabel("time step t"); ax.legend()
axes[0].set_ylabel("gradient norm (mean over 256 sequences)")
plt.tight_layout(); plt.savefig(FIG / "grad_norm_vs_t_T100.png", dpi=160); plt.show()

print(pd.DataFrame({k: {"|grad| at t=1 / t=T (after)": PROF[(k,'after')][0] / PROF[(k,'after')][-1],
                        "|grad| at t=T (after)": PROF[(k,'after')][-1]} for k in ["RNN","LSTM","GRU"]}).T)

**(b) Hidden-size sweep** for LSTM and GRU on HAR, `H ∈ {16, 32, 64, 128}`. H = 64 is taken from the main runs in Task 3 (same settings).

In [ ]:
SWEEP_H = [16, 32, 64, 128]
SWEEP_SEEDS = [0]                 # one seed for the sweep to save time (H=64 uses seed 0 too)
for kind in ["LSTM", "GRU"]:
    for hh in SWEEP_H:
        if hh == H:
            continue
        for s in SWEEP_SEEDS:
            HAR_RES[(kind, hh, s)] = train_har(kind, hh, s, note=f"; sweep")
            r = HAR_RES[(kind, hh, s)]
            print(f"sweep {kind:4s} H={hh:3d} seed {s}: test acc {r['acc']:.4f}  params {r['params']}  {r['sec']:.2f}s/epoch")

rows = []
for kind in ["LSTM","GRU"]:
    for hh in SWEEP_H:
        rs = [HAR_RES[(kind, hh, s)] for s in SWEEP_SEEDS]
        acc = np.array([r["acc"] for r in rs])
        rows.append({"model": kind, "H": hh, "params (total)": rs[0]["params"],
                     "test acc (%) seed 0": f"{100*acc.mean():.2f}",
                     "sec/epoch": round(float(np.mean([r['sec'] for r in rs])), 2)})
sweep_table = pd.DataFrame(rows)
display(sweep_table)
sweep_table.to_csv(FIG.parent / "hidden_sweep_table.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
for kind in ["LSTM","GRU"]:
    mu = [100*np.mean([HAR_RES[(kind, hh, s)]["acc"] for s in SWEEP_SEEDS]) for hh in SWEEP_H]
    ax.plot(SWEEP_H, mu, marker="o", label=kind)
ax.set_xscale("log", base=2); ax.set_xticks(SWEEP_H, [str(h) for h in SWEEP_H])
ax.set_xlabel("hidden size H"); ax.set_ylabel("HAR test accuracy (%)"); ax.legend()
ax.set_title("Hidden-size sweep on HAR (seed 0)")
plt.tight_layout(); plt.savefig(FIG / "hidden_sweep_har.png", dpi=160); plt.show()

**(c) Individual ablation.** The brief assigns one of these by the last digit of the student ID. The student ID is not known to this notebook, so both are run at T = 200, three seeds each:
- even last digit → gradient clipping (norm 1.0 vs none) for the vanilla RNN;
- odd last digit → LSTM forget-gate bias 0 vs 1.

The clip = 1.0 RNN runs are the Task 4 main runs (identical settings), so they are reused.

In [ ]:
ABL = {"rnn_clip": {}, "lstm_forget": {}}
for s in SEEDS:
    ABL["rnn_clip"][("clip 1.0", s)] = ADD_RES[("RNN", 200, s)]
    ABL["rnn_clip"][("no clip", s)] = train_adding("RNN", 200, s, clip=None, note="; ablation=no_clip")
    ABL["lstm_forget"][("forget bias 0", s)] = train_adding("LSTM", 200, s, forget_bias=0.0, note="; ablation=forget_bias_0")
    ABL["lstm_forget"][("forget bias 1", s)] = train_adding("LSTM", 200, s, forget_bias=1.0, note="; ablation=forget_bias_1")

abl_rows = []
for name, group in ABL.items():
    for cond in sorted({c for c, _ in group}):
        m = np.array([group[(cond, s)]["mse"] for s in SEEDS])
        abl_rows.append({"ablation": name, "condition": cond, "test MSE": f"{np.nanmean(m):.4f} ± {np.nanstd(m, ddof=1):.4f}",
                         "solved (of 3)": sum(group[(cond, s)]["solved"] for s in SEEDS)})
abl_table = pd.DataFrame(abl_rows)
display(abl_table)
abl_table.to_csv(FIG.parent / "ablation_table.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, title) in zip(axes, [("rnn_clip", "vanilla RNN, T = 200: gradient clipping"),
                                    ("lstm_forget", "LSTM, T = 200: forget-gate bias")]):
    conds = sorted({c for c, _ in ABL[name]})
    vals = [[ABL[name][(c, s)]["mse"] for s in SEEDS] for c in conds]
    ax.boxplot(vals, tick_labels=conds)
    for i, v in enumerate(vals):
        ax.scatter([i+1]*len(v), v, color="k", zorder=3, s=14)
    ax.axhline(0.05, color="grey", ls=":"); ax.set_title(title); ax.set_ylabel("test MSE")
plt.tight_layout(); plt.savefig(FIG / "ablation_T200.png", dpi=160); plt.show()

In [ ]:
save_results()
print("Rows written:", len(RESULTS), "->", RESULTS_PATH.resolve())
print(pd.read_csv(RESULTS_PATH).groupby(["model","seq_len"], dropna=False).size().head(20))